# ReAct + 실용 도구 묶음
- 도구 3~5개로 진짜 쓸 만한 ReAct 에이전트. 

## 환경 준비

`.env` 는 상위 폴더들과 동일하게 공유.

```
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=mtvs2026-langgraph
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

## 1. 도구 5개 정의

- 각 도구는 가짜 데이터를 쓰지만 실제 응용에서는 외부 API · DB · 파일 시스템으로 교체.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송/처리 상태를 조회한다."""
    db = {
        "O-1001": "배송 지연 5일. 제주 물류센터에서 대기 중.",
        "O-1002": "배송 준비 중. 내일 출고 예정.",
        "O-1003": "배송 완료. 고객 수령 확인됨.",
        "O-1004": "반품 접수 완료. 환불 심사 대기 중.",
    }
    return db.get(order_id, f"{order_id} 주문 정보 없음")


@tool
def calculate_refund_amount(order_amount: int, used_coupon: int = 0, opened: bool = False) -> str:
    """주문금액, 사용 쿠폰, 개봉 여부로 환불 가능 금액을 계산한다."""
    if opened:
        return "개봉 상품: 단순 변심 환불 불가. 하자 여부 확인 필요"
    refund = max(order_amount - used_coupon, 0)
    return f"환불 가능 금액: {refund}원"


@tool
def calculate_delay_compensation(delay_days: int, order_amount: int) -> str:
    """배송 지연 일수와 주문 금액으로 보상 쿠폰 금액을 계산한다."""
    if delay_days < 3:
        return "보상 대상 아님"
    coupon = min(int(order_amount * 0.1), 10000)
    return f"배송 지연 보상 쿠폰: {coupon}원"


@tool
def search_policy(keyword: str) -> str:
    """고객지원 정책 정보 검색."""
    policies = {
        "배송 지연": "출고 예정일보다 3일 이상 지연되면 주문금액의 10%, 최대 1만원 쿠폰 보상.",
        "전자제품": "미개봉 전자제품은 수령 후 7일 이내 환불 가능. 개봉 상품은 하자 확인 필요.",
        "식품": "신선식품은 단순 변심 환불 불가. 파손/오배송은 사진 확인 후 처리.",
        "쿠폰": "주문 취소 시 사용 쿠폰은 유효기간 내 자동 복구. 일부 프로모션 쿠폰 제외.",
    }
    for k, v in policies.items():
        if k in keyword:
            return v
    return "관련 정책 정보 없음"


@tool
def draft_reply_template(issue_type: str) -> str:
    """문의 유형(배송 지연/환불/교환)에 맞는 고객 답변 템플릿을 제안한다."""
    templates = {
        "배송 지연": "불편을 드려 죄송합니다. 현재 배송 상태를 확인했으며 지연 사유와 예상 일정을 안내드리겠습니다.",
        "환불": "환불 가능 여부와 예상 환불 금액을 확인해 안내드립니다. 결제수단별 처리 기간도 함께 확인해드리겠습니다.",
        "교환": "교환 가능 조건을 확인한 뒤 회수 접수와 재출고 일정을 안내드리겠습니다.",
    }
    return templates.get(issue_type, "문의 내용을 확인한 뒤 필요한 조치와 예상 일정을 안내드리겠습니다.")


tools= 


## 2. ReAct 에이전트, 한 번에 여러 도구 연쇄

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
llm = 
llm_tools = 

In [ ]:
def call_llm(state: MessagesState) -> dict:
    # 시스템 프롬프트로 일관된 톤 유지
    msgs = [SystemMessage("너는 이커머스 고객지원 티켓 처리를 돕는 한국어 비서. 필요하면 도구를 적극적으로 활용해.")] + state["messages"]
    return {"messages": [llm_tools.invoke(msgs)]}

In [ ]:
graph = 
graph.add_node("llm", )
graph.add_node("tools", )
graph.add_edge()
graph.add_conditional_edges()
graph.add_edge()
app =

## 3. 시나리오 A, 배송 지연 + 환불 계산 (도구 3개 연쇄)


In [ ]:
q = "O-1001 주문 상태를 확인하고, 주문금액 120000원에 쿠폰 10000원을 쓴 미개봉 전자제품의 환불 가능 금액과 배송 지연 5일 보상 쿠폰을 계산해줘."

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("Q:", q)
print()
print("=== 도구 호출 흐름 ===")
for m in result["messages"]:
    name = type(m).__name__
    if name == "AIMessage" and m.tool_calls:
        for call in m.tool_calls:
            print(f"  [Act]     {call['name']}({call['args']})")
    elif name == "ToolMessage":
        print(f"  [Observe] {m.content}")
print()
print("최종 답:", result["messages"][-1].content)


## 4. 시나리오 B, 정책 검색 + 답변 템플릿 (도구 2개)


In [ ]:
q = " "

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("최종 답:", result["messages"][-1].content)


## 5. 시나리오 C, 여러 주문 상태 조회 (도구 1개 여러 번)


In [ ]:
q = " "

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("최종 답:", result["messages"][-1].content)


## 6. 흐름 디버깅, `app.stream` 으로 노드별 실시간 출력

In [ ]:
print("실시간 노드 호출 추적:")
for chunk in app.stream(
    {"messages": [HumanMessage("O-1002 주문 상태를 확인하고 배송 지연 정책과 답변 템플릿을 참고해 고객 답변을 준비해줘.")]},
    config={"recursion_limit": 15},
    stream_mode="updates",
):
    for node_name, payload in chunk.items():
        last = payload["messages"][-1]
        kind = type(last).__name__
        body = str(last.content)[:70] if last.content else f"(tool_calls: {[c['name'] for c in getattr(last, 'tool_calls', [])]})"
        print(f"  [{node_name}] {kind}: {body}")


## 7. 정리

- 도구는 짧은 docstring 으로 모델에게 사용법 안내. **이름 + 설명이 명확할수록** ReAct 가 잘 작동
- `recursion_limit` 으로 무한 루프 방지 (10~20 권장)
- `app.stream(stream_mode="updates")` 로 단계별 진행 디버깅
- 도구 5개 정도면 충분. 너무 많으면 LLM 이 헷갈림

## [실습]

1. 서비스 도구 확장하기: `draft_reply_template`에 `결제 오류` 유형을 추가하고, `lookup_customer_tier(customer_id)` 도구를 새로 붙이세요.
2. 도구 호출 흐름 디버깅하기: ReAct 그래프의 Act/Observe를 출력하고, 한 질문에서 도구가 몇 번 호출됐는지 집계하세요.
3. 프롬프트와 안전장치 비교하기: 같은 질문을 "도구 적극 사용" 프롬프트와 "도구 사용 금지" 프롬프트로 실행하고, `recursion_limit`을 낮췄을 때의 차이를 확인하세요.
